In [1]:
# Imports
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import os
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import timm
from torch.optim import AdamW
from tqdm import tqdm
from pathlib import Path

/home/cqilab/anaconda3/envs/llmfinetune/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Hyperparams
BATCH_SIZE = 8
NUM_EPOCHS = 150
LEARNING_RATE = 0.0005
NUM_CLASSES = 83

device = torch.device(f"cuda:1" if torch.cuda.is_available() else "cpu")

# Dataset path
csv_path = "./Dataset/vit_dataset.csv"

# ---------- label maps ----------
TYPE2IDX = {
    "Cat state":      0,
    "Coherent state": 1,
    "Thermal state":  2,
    "Fock state":     3,
    "Random state":   4,
    "Number state":   5,
}
#  6 + 30 + 16 + 14 + 11 + 6

# qubit values assumed to be 1‒30  ➜ map «value → class‑idx»
QUBIT2IDX = {q: (q - 1) for q in range(0, 31)}      # 30 classes (0‑29)

# alpha values assumed to be 0‒15  ➜ map «value → class‑idx»
ALPHA2IDX = {a: a for a in range(0, 16)}      # 16 classes (0‑15)

# photons values assumed to be 3‒15  ➜ map «value → class‑idx»
PHOTONS2IDX = {p: (p - 2) for p in range(3, 16)}      # 14 classes (0‑13)
PHOTONS2IDX[0] = 0

DENSITY2IDX = {(float(d/10)): d for d in range(0,11)}

LINSPACE2IDX = {l: (l-5) for l in range(5,11)}

# Dataset class multiclass classification
class WignerDataset(Dataset):
    def __init__(self, csv_file=None, dataframe=None, image_dir=None, transform=None):
        """
        Args:
            csv_file (str): Path to the CSV file
            image_dir (str): Root dir to prepend to image path if not included in CSV
            transform (callable, optional): Transform to apply on images (e.g., Resize, ToTensor)
        """
        if dataframe is not None:
            self.data = dataframe.reset_index(drop=True)
        elif csv_file is not None:
            self.data = pd.read_csv(csv_file)
        else:
            raise ValueError("Either csv_file or dataframe must be provided.")
        
        self.image_dir = image_dir
        self.transform = transform


    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        # Image loading
        img_path = row['image']
        if self.image_dir and not os.path.isabs(img_path):
            img_path = os.path.join(self.image_dir, img_path)
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        # Labels (modify depending on task)
        label = {
            'type': TYPE2IDX[row['type']],  # convert class label to int
            'number_of_qubit': QUBIT2IDX[row['number_of_qubit']],
            'alpha': ALPHA2IDX[row['alpha']],
            'number_of_photons': PHOTONS2IDX[row['number_of_photons']],
            'density': DENSITY2IDX[row['density']],
            'linspace': LINSPACE2IDX[row['linspace']],
        }

        return image, label

In [3]:
# Transformations
transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


# Train test splitfrom sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(pd.read_csv(csv_path), test_size=0.2, random_state=42)
# train_df, test_df = train_test_split(pd.read_csv(csv_path), test_size=0.9, random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=42)

train_data = WignerDataset(dataframe=train_df, image_dir="./Dataset/", transform=transforms)
val_data = WignerDataset(dataframe=val_df, image_dir="./Dataset/", transform=transforms)
test_data = WignerDataset(dataframe=test_df, image_dir="./Dataset/", transform=transforms)


# dataset = WignerDataset(csv_path, transform=transforms)

# Train dataLoader and test dataLoader
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

In [4]:
# System model using ViT-large-16-224
# Use timm
# Last layer should be 70
# Task is multiclass classification
# Use partition of nodes to determine each class, 0-6 (what highest prob), 7-36 (what highest prob), 37-47 (what highest prob), 
# 49-59 (what highest prob), 59-69 (what highest prob)

class ViTMultiClassPartitioned(nn.Module):
    def __init__(self, num_classes=83):
        super(ViTMultiClassPartitioned, self).__init__()
        # Load ViT-Large-16-224 pretrained from timm
        self.backbone = timm.create_model("vit_large_patch16_224", pretrained=True)

        # Add new classification head
        self.classifier = nn.Sequential(
            nn.Linear(1000, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.backbone(x)
        logits = self.classifier(x)
        return logits

    def partition_predictions(self, logits):
        """
        Partition logic:
        - [0-5]       -> Type (6)
        - [6-35]      -> Number of qubits (30)
        - [36-51]     -> Alpha (16)
        - [52-65]     -> Number of photons (14)
        - [66-76]     -> Density (11)
        - [77-82]     -> Linear Space (6)
        Returns dict with max-predicted class index for each partition
        """
        preds = {}
        preds['type'] = logits[:, 0:6]
        preds['number_of_qubit'] = logits[:, 6:36]
        preds['alpha'] = logits[:, 36:52]
        preds['number_of_photons'] = logits[:, 52:66]
        preds['density'] = logits[:, 66:77]
        preds['linspace'] = logits[:, 77:83]
        return preds


In [5]:
model = ViTMultiClassPartitioned(num_classes=NUM_CLASSES).to(device)

# Loss and optimizer (AdamW), MSE loss & CrossEntropy loss
criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

# Checkpoint
ckpt_dir = Path("./checkpoints")
ckpt_dir.mkdir(exist_ok=True)
best_acc = 0.0        # highest val accuracy seen so far

# Scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=3, verbose=True
)

best_ckpt = ckpt_dir / "best.ckpt"
if best_ckpt.exists():
    ckpt = torch.load(best_ckpt, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optim"])
    scheduler.load_state_dict(ckpt["sched"])
    best_acc = ckpt["val_acc"]
    start_epoch = ckpt["epoch"] + 1
    print(f"✓ Resumed from epoch {start_epoch} with best_acc={best_acc:.2f}%")
else:
    start_epoch = 0

/home/cqilab/anaconda3/envs/llmfinetune/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [6]:
# Training loop, perform test for every 10 epochs, calculate accuracy for multiclass classification (each partition)

for epoch in range(start_epoch, NUM_EPOCHS):
    model.train()
    running_loss = 0.0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}"):
        images = images.to(device)

        optimizer.zero_grad()
        logits = model(images)
        
        preds = model.partition_predictions(logits)

        loss_total = 0.0

        # type: 0–5
        preds_type = preds['type'] # [0-5]
        targets_type = labels['type'].to(device) # [3]
        # print(f"DEBUGGING")
        # ############# DEBUGGING #############
        
        # # print preds_type check type and targets_type check type 
        # print(f"preds_type: {preds_type}, targets_type: {targets_type}")
        # # check if preds_type and targets_type are same shape
        # print(f"preds_type shape: {preds_type.shape}, targets_type shape: {targets_type.shape}")
        # # check if preds_type and targets_type are same dtype
        # print(f"preds_type dtype: {preds_type.dtype}, targets_type dtype: {targets_type.dtype}")
        
        loss_type = criterion(preds_type, targets_type)
        loss_total += loss_type

        # number_of_qubit: 6-35
        preds_qubit = preds['number_of_qubit']
        targets_qubit = labels['number_of_qubit'].to(device)
        loss_qubit = criterion(preds_qubit, targets_qubit)
        loss_total += loss_qubit

        # alpha: 36-51
        preds_alpha = preds['alpha']
        targets_alpha = labels['alpha'].to(device)
        loss_alpha = criterion(preds_alpha, targets_alpha)
        loss_total += loss_alpha

        # number_of_photons: 52-65
        preds_photon = preds['number_of_photons']
        targets_photon = labels['number_of_photons'].to(device)
        loss_photon = criterion(preds_photon, targets_photon)
        loss_total += loss_photon

        # density: 66-76
        preds_density = preds['density']
        targets_density = labels['density'].to(device)
        loss_density = criterion(preds_density, targets_density)
        loss_total += loss_density
            
        # linspace: 77-82
        preds_linspace = preds['linspace']
        targets_linspace = labels['linspace'].to(device)
        loss_linspace = criterion(preds_linspace, targets_linspace)
        loss_total += loss_linspace

        loss_total.backward()
        optimizer.step()
        running_loss += loss_total.item()

    print(f"[Epoch {epoch+1}] Train Loss: {running_loss / len(train_loader):.4f}")

     # ✅ Evaluate every 2 epochs
    if (epoch) % 5 == 0:
        model.eval()

        correct_total = 0
        total_total = 0

        # Partition-wise tracking
        partition_correct = [0] * 6
        partition_total = [0] * 6

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                logits = model(images)
                
                preds = model.partition_predictions(logits)

                # type: 0–5
                preds_type = preds['type'].argmax(1)
                targets_type = labels['type'].to(device)
                correct = (preds_type == targets_type).sum().item()
                partition_correct[0] += correct
                partition_total[0] += BATCH_SIZE
                

                # number_of_qubit: 6-35
                preds_type = preds['number_of_qubit'].argmax(1)
                targets_type = labels['number_of_qubit'].to(device)
                correct = (preds_type == targets_type).sum().item()
                partition_correct[1] += correct
                partition_total[1] += BATCH_SIZE

                
                # alpha: 36-51
                preds_alpha = preds['alpha'].argmax(1)
                targets_alpha = labels['alpha'].to(device)
                correct = (preds_alpha == targets_alpha).sum().item()
                partition_correct[2] += correct
                partition_total[2] += BATCH_SIZE

                
                # number_of_photons: 52-65
                preds_photon = preds['number_of_photons'].argmax(1)
                targets_photon = labels['number_of_photons'].to(device)
                correct = (preds_photon == targets_photon).sum().item()
                partition_correct[3] += correct
                partition_total[3] += BATCH_SIZE
                
                # density: 66-76
                preds_density = preds['density'].argmax(1)
                targets_density = labels['density'].to(device)
                correct = (preds_density == targets_density).sum().item()
                partition_correct[4] += correct
                partition_total[4] += BATCH_SIZE

                # linspace: 77-82
                preds_linspace = preds['linspace'].argmax(1)
                targets_linspace = labels['linspace'].to(device)
                correct = (preds_linspace == targets_linspace).sum().item()
                partition_correct[5] += correct
                partition_total[5] += BATCH_SIZE

        # Compute per-partition accuracy
        print(f"\n🧪 [Epoch {epoch+1}] Partitioned Accuracy:")
        for i in range(6):
            if partition_total[i] > 0:
                acc = 100. * partition_correct[i] / partition_total[i]
                print(f"  Partition {i+1}: {acc:.2f}% ({partition_correct[i]}/{partition_total[i]})")
            else:
                print(f"  Partition {i+1}: No samples")

        # Total accuracy
        correct_total = sum(partition_correct)
        total_total = sum(partition_total)
        total_acc = 100. * correct_total / total_total
        print(f"  ➤ Overall Accuracy: {total_acc:.2f}%")
        
        # ►► step LR scheduler & save checkpoints ◄◄
        scheduler.step(total_acc)                    # adjust LR if plateau
        
        # save rolling checkpoint every evaluation
        torch.save(
            {
                "epoch": epoch,
                "model": model.state_dict(),
                "optim": optimizer.state_dict(),
                "sched": scheduler.state_dict(),
                "val_acc": total_acc,
            },
            ckpt_dir / "last.ckpt",
        )

        # save the best checkpoint
        if total_acc > best_acc:
            best_acc = total_acc
            torch.save(
                {
                    "epoch": epoch,
                    "model": model.state_dict(),
                    "optim": optimizer.state_dict(),
                    "sched": scheduler.state_dict(),
                    "val_acc": best_acc,
                },
                ckpt_dir / "best.ckpt",
            )
            print(f"  ✔ New best model saved  (val_acc = {best_acc:.2f}%)")

Epoch 1/150:   0%|          | 0/883 [00:00<?, ?it/s]

Epoch 1/150: 100%|██████████| 883/883 [15:20<00:00,  1.04s/it]


[Epoch 1] Train Loss: 9.6633

🧪 [Epoch 1] Partitioned Accuracy:
  Partition 1: 78.11% (1381/1768)
  Partition 2: 3.85% (68/1768)
  Partition 3: 53.05% (938/1768)
  Partition 4: 66.40% (1174/1768)
  Partition 5: 84.90% (1501/1768)
  Partition 6: 17.93% (317/1768)
  ➤ Overall Accuracy: 50.71%
  ✔ New best model saved  (val_acc = 50.71%)


Epoch 2/150: 100%|██████████| 883/883 [12:49<00:00,  1.15it/s]


[Epoch 2] Train Loss: 8.3025


...

[Epoch 148] Train Loss: 1.7304


Epoch 149/150: 100%|██████████| 883/883 [12:43<00:00,  1.16it/s]


[Epoch 149] Train Loss: 1.6699


Epoch 150/150: 100%|██████████| 883/883 [12:43<00:00,  1.16it/s]

[Epoch 150] Train Loss: 1.7453


In [7]:
counter = 0
for images, labels in tqdm(train_loader, desc=f"Epoch {0+1}/{NUM_EPOCHS}"):
    if counter  == 1:
        break
    
    images = images.to(device)

    optimizer.zero_grad()
    logits = model(images)
    
    preds = model.partition_predictions(logits)

    loss_total = 0.0

    # type: 0–5
    preds_type = preds['type'] # [0-5]
    targets_type = labels['type'].to(device) # [3]
    loss_type = criterion(preds_type, targets_type)
    loss_total += loss_type
    
    print(f"DEBUGGING")
    ############# DEBUGGING #############
    
    # print preds_type check type and targets_type check type 
    print(f"preds_type: {preds_type}, targets_type: {targets_type}")
    # check if preds_type and targets_type are same shape
    print(f"preds_type shape: {preds_type.shape}, targets_type shape: {targets_type.shape}")
    # check if preds_type and targets_type are same dtype
    print(f"preds_type dtype: {preds_type.dtype}, targets_type dtype: {targets_type.dtype}")

    # number_of_qubit: 6-35
    preds_qubit = preds['number_of_qubit']
    targets_qubit = labels['number_of_qubit'].to(device)
    loss_qubit = criterion(preds_qubit, targets_qubit)
    loss_total += loss_qubit
    
    print(f"DEBUGGING")
    ############# DEBUGGING #############
    
    # print preds_type check type and targets_type check type 
    print(f"preds_type: {preds_type}, targets_type: {targets_type}")
    # check if preds_type and targets_type are same shape
    print(f"preds_type shape: {preds_type.shape}, targets_type shape: {targets_type.shape}")
    # check if preds_type and targets_type are same dtype
    print(f"preds_type dtype: {preds_type.dtype}, targets_type dtype: {targets_type.dtype}")

    # alpha: 36-51
    preds_alpha = preds['alpha']
    targets_alpha = labels['alpha'].to(device)
    loss_alpha = criterion(preds_alpha, targets_alpha)
    loss_total += loss_alpha
    
    print(f"DEBUGGING")
    ############# DEBUGGING #############
    
    # print preds_alpha check type and targets_alpha check type 
    print(f"preds_alpha: {preds_alpha}, targets_alpha: {targets_alpha}")
    # check if preds_alpha and targets_alpha are same shape
    print(f"preds_alpha shape: {preds_alpha.shape}, targets_alpha shape: {targets_alpha.shape}")
    # check if preds_alpha and targets_alpha are same dtype
    print(f"preds_alpha dtype: {preds_alpha.dtype}, targets_alpha dtype: {targets_alpha.dtype}")

    # number_of_photons: 52-65
    preds_photon = preds['number_of_photons']
    targets_photon = labels['number_of_photons'].to(device)
    loss_photon = criterion(preds_photon, targets_photon)
    loss_total += loss_photon
    
    print(f"DEBUGGING")
    ############# DEBUGGING #############
    
    # print preds_photon check type and targets_photon check type 
    print(f"preds_photon: {preds_photon}, targets_photon: {targets_photon}")
    # check if preds_photon and targets_photon are same shape
    print(f"preds_photon shape: {preds_photon.shape}, targets_photon shape: {targets_photon.shape}")
    # check if preds_photon and targets_photon are same dtype
    print(f"preds_photon dtype: {preds_photon.dtype}, targets_photon dtype: {targets_photon.dtype}")


    # density: 66-76
    preds_density = preds['density']
    targets_density = labels['density'].to(device)
    loss_density = criterion(preds_density, targets_density)
    loss_total += loss_density
    
    print(f"DEBUGGING")
    ############# DEBUGGING #############
    
    # print preds_density check type and targets_density check type 
    print(f"preds_density: {preds_density}, targets_density: {targets_density}")
    # check if preds_density and targets_density are same shape
    print(f"preds_density shape: {preds_density.shape}, targets_density shape: {targets_density.shape}")
    # check if preds_density and targets_density are same dtype
    print(f"preds_density dtype: {preds_density.dtype}, targets_density dtype: {targets_density.dtype}")
        
    # linspace: 77-82
    preds_linspace = preds['linspace']
    targets_linspace = labels['linspace'].to(device)
    loss_linspace = criterion(preds_linspace, targets_linspace)
    loss_total += loss_linspace
    
    print(f"DEBUGGING preds_linspace")
    ############# DEBUGGING #############
    
    # print preds_linspace check type and targets_linspace check type 
    print(f"preds_linspace: {preds_linspace}, targets_linspace: {targets_linspace}")
    # check if preds_linspace and targets_linspace are same shape
    print(f"preds_linspace shape: {preds_linspace.shape}, targets_linspace shape: {targets_linspace.shape}")
    # check if preds_linspace and targets_linspace are same dtype
    print(f"preds_linspace dtype: {preds_linspace.dtype}, targets_linspace dtype: {targets_linspace.dtype}")

    loss_total.backward()
    optimizer.step()
    running_loss += loss_total.item()

    print(f"[Epoch {epoch+1}] Train Loss: {running_loss / len(train_loader):.4f}")
    
    counter += 1

    # # ✅ Evaluate every 1 epochs
    # if (epoch) % 5 == 0:
    #     model.eval()

    #     correct_total = 0
    #     total_total = 0

    #     # Partition-wise tracking
    #     partition_correct = [0] * 6
    #     partition_total = [0] * 6

    #     with torch.no_grad():
    #         for images, labels in train_loader:
    #             images = images.to(device)
    #             logits = model(images)
                
    #             preds = model.partition_predictions(logits)
                
    #             bs = images.size(0)                 # number of images in this batch

    #             # type: 0–5
    #             preds_type = preds['type'].argmax(1)
    #             targets_type = labels['type'].to(device)
    #             correct = (preds_type == targets_type).sum().item()
    #             partition_correct[0] += correct
    #             partition_total[0] += bs
                

    #             # number_of_qubit: 6-35
    #             preds_type = preds['number_of_qubit'].argmax(1)
    #             targets_type = labels['number_of_qubit'].to(device)
    #             correct = (preds_type == targets_type).sum().item()
    #             partition_correct[1] += correct
    #             partition_total[1] += bs

                
    #             # alpha: 36-51
    #             preds_alpha = preds['alpha'].argmax(1)
    #             targets_alpha = labels['alpha'].to(device)
    #             correct = (preds_alpha == targets_alpha).sum().item()
    #             partition_correct[2] += correct
    #             partition_total[2] += bs

                
    #             # number_of_photons: 52-65
    #             preds_photon = preds['number_of_photons'].argmax(1)
    #             targets_photon = labels['number_of_photons'].to(device)
    #             correct = (preds_photon == targets_photon).sum().item()
    #             partition_correct[3] += correct
    #             partition_total[3] += bs
                
    #             # density: 66-76
    #             preds_density = preds['density'].argmax(1)
    #             targets_density = labels['density'].to(device)
    #             correct = (preds_density == targets_density).sum().item()
    #             partition_correct[4] += correct
    #             partition_total[4] += bs

    #             # linspace: 77-82
    #             preds_linspace = preds['linspace'].argmax(1)
    #             targets_linspace = labels['linspace'].to(device)
    #             correct = (preds_linspace == targets_linspace).sum().item()
    #             partition_correct[5] += correct
    #             partition_total[5] += bs

    #     # Compute per-partition accuracy
    #     print(f"\n🧪 [Epoch {epoch+1}] Partitioned Accuracy:")
    #     for i in range(5):
    #         if partition_total[i] > 0:
    #             acc = 100. * partition_correct[i] / partition_total[i]
    #             print(f"  Partition {i+1}: {acc:.2f}% ({partition_correct[i]}/{partition_total[i]})")
    #         else:
    #             print(f"  Partition {i+1}: No samples")

    #     # Total accuracy
    #     correct_total = sum(partition_correct)
    #     total_total = sum(partition_total)
    #     total_acc = 100. * correct_total / total_total
    #     print(f"  ➤ Overall Accuracy: {total_acc:.2f}%")

Epoch 1/150:   0%|          | 0/883 [00:00<?, ?it/s]

DEBUGGING
preds_type: tensor([[-19.9554, -10.9414, -83.7935,  -6.5868,  12.0024, -48.8671],
        [-14.2961,  -0.5063, -38.1326, -14.2557, -13.6154, -25.3038],
        [-33.6728,  -6.6806,   8.7484, -27.8225, -13.1853, -18.2347],
        [-45.7319, -10.0252,   9.8519, -40.5371, -11.7061, -19.3075],
        [-56.8573, -30.5814,  31.3532, -44.8012, -29.0662,  -3.0417],
        [-37.6467,   8.7032, -53.2454, -68.4983, -21.7810, -34.0945],
        [ -6.4512,  -1.9780, -11.3874,   9.8376,  -2.7852, -21.4738],
        [-13.8031,  -3.3221,  -6.2994,  11.6708,  -5.0157, -23.3573]],
       device='cuda:1', grad_fn=<SliceBackward0>), targets_type: tensor([4, 1, 2, 2, 2, 1, 3, 3], device='cuda:1')
preds_type shape: torch.Size([8, 6]), targets_type shape: torch.Size([8])
preds_type dtype: torch.float32, targets_type dtype: torch.int64
DEBUGGING
preds_type: tensor([[-19.9554, -10.9414, -83.7935,  -6.5868,  12.0024, -48.8671],
        [-14.2961,  -0.5063, -38.1326, -14.2557, -13.6154, -25.3038],
 

Epoch 1/150:   0%|          | 1/883 [00:01<15:32,  1.06s/it]

[Epoch 150] Train Loss: 1.7471


Epoch 1/150:   0%|          | 1/883 [00:01<25:06,  1.71s/it]
